# Part C - Anomaly detection and K-Means Segmentation


In [4]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score

RANDOM_STATE = 42

if not os.path.exists('outputs'):
    os.makedirs('outputs')

In [6]:
txn = pd.read_csv("/content/txn_behaviour.csv")

In [7]:
behaviour_features = ["txn_hour", "is_new_device", "txn_amount_inr"]

X_behaviour = txn[behaviour_features].copy()
scaler_b = StandardScaler()
X_behaviour_scaled = scaler_b.fit_transform(X_behaviour)

n_total = len(txn)
n_seeded = int(txn["txn_id"].str.startswith("BTXNA").sum())
contamination = n_seeded / n_total
print(f"Seeded anomaly proportion: {n_seeded}/{n_total} = {contamination:.4f}")

Seeded anomaly proportion: 15/265 = 0.0566


In [8]:
iso = IsolationForest(random_state=RANDOM_STATE, contamination=contamination)
iso_pred = iso.fit_predict(X_behaviour_scaled)   # -1 = anomaly, 1 = normal
txn["iso_flagged_anomaly"] = (iso_pred == -1).astype(int)

is_seeded = txn["txn_id"].str.startswith("BTXNA")
n_flagged_total = int(txn["iso_flagged_anomaly"].sum())
n_seeded_flagged = int(txn.loc[is_seeded, "iso_flagged_anomaly"].sum())
recall = n_seeded_flagged / n_seeded

In [11]:
print(f"Total flagged as anomalous by IsolationForest: {n_flagged_total} / {n_total}")
print(f"Of the {n_seeded} seeded BTXNA* anomalies, {n_seeded_flagged} were flagged.")
print(f"Recall against seeded ground truth: {recall:.4f} ({n_seeded_flagged}/{n_seeded})")
txn.to_csv("outputs/txn_with_anomaly_flags.csv", index=False)

part_c_report = {
    "n_txn_total": n_total,
    "n_seeded_anomalies": n_seeded,
    "contamination_used": round(contamination, 4),
    "n_flagged_total": n_flagged_total,
    "n_seeded_flagged": n_seeded_flagged,
    "recall_on_seeded_anomalies": round(recall, 4),
}

Total flagged as anomalous by IsolationForest: 15 / 265
Of the 15 seeded BTXNA* anomalies, 11 were flagged.
Recall against seeded ground truth: 0.7333 (11/15)


In [12]:
# quick scatter viz: txn_amount vs txn_hour, colored by flag, marker for seeded truth
plt.figure(figsize=(7, 5))
normal_mask = txn["iso_flagged_anomaly"] == 0
plt.scatter(txn.loc[normal_mask, "txn_hour"], txn.loc[normal_mask, "txn_amount_inr"],
            c="steelblue", alpha=0.5, label="Flagged normal", s=25)
flagged_mask = txn["iso_flagged_anomaly"] == 1
plt.scatter(txn.loc[flagged_mask, "txn_hour"], txn.loc[flagged_mask, "txn_amount_inr"],
            c="orange", alpha=0.8, label="Flagged anomalous", s=35)
seeded_mask = txn["txn_id"].str.startswith("BTXNA")
plt.scatter(txn.loc[seeded_mask, "txn_hour"], txn.loc[seeded_mask, "txn_amount_inr"],
            facecolors="none", edgecolors="red", s=90, linewidths=1.5, label="Seeded ground-truth (BTXNA*)")
plt.xlabel("Transaction hour")
plt.ylabel("Transaction amount (INR)")
plt.title("IsolationForest anomaly flags vs seeded ground truth")
plt.legend()
plt.tight_layout()
plt.savefig("outputs/isolation_forest_scatter.png", dpi=150)
plt.close()

# Optional: K-Means segmentation

In [15]:
df = pd.read_csv("/content/credit_applicants.csv")
segment_features = [
    "age", "monthly_income_inr", "existing_loans_count",
    "credit_utilization_ratio", "upi_monthly_inflow_inr",
    "bounced_payments_count",
]

X_seg = df[segment_features].copy()
scaler_s = StandardScaler()
X_seg_scaled = scaler_s.fit_transform(X_seg)

In [16]:
ch_scores = {}
inertias = {}
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_seg_scaled)
    ch_scores[k] = calinski_harabasz_score(X_seg_scaled, labels)
    inertias[k] = km.inertia_

best_k = max(ch_scores, key=ch_scores.get)
print(f"\nCalinski-Harabasz scores by k: {ch_scores}")
print(f"Best k by Calinski-Harabasz index: {best_k}")


Calinski-Harabasz scores by k: {2: np.float64(63.914985653118016), 3: np.float64(57.672957261837006), 4: np.float64(56.071175802181756), 5: np.float64(53.90961854964549), 6: np.float64(50.70717039368732), 7: np.float64(49.90194800694481)}
Best k by Calinski-Harabasz index: 2


In [17]:
# elbow plot
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(inertias.keys()), list(inertias.values()), marker="o")
axes[0].set_title("Elbow method (inertia)")
axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia")
axes[1].plot(list(ch_scores.keys()), list(ch_scores.values()), marker="o", color="darkorange")
axes[1].set_title("Calinski-Harabasz index")
axes[1].set_xlabel("k"); axes[1].set_ylabel("CH score")
axes[1].axvline(best_k, linestyle="--", color="gray")
plt.tight_layout()
plt.savefig("outputs/kmeans_k_selection.png", dpi=150)
plt.close()

In [18]:
km_final = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
df["segment"] = km_final.fit_predict(X_seg_scaled)

segment_default_rates = df.groupby("segment")["default"].agg(["size", "mean"]).rename(
    columns={"size": "n_applicants", "mean": "default_rate"}
)
segment_default_rates["default_rate"] = segment_default_rates["default_rate"].round(4)
overall_rate = df["default"].mean()
print(f"\nOverall default rate: {overall_rate:.4f}")
print("Per-segment default rate:")
print(segment_default_rates.to_string())

over_indexing = segment_default_rates[segment_default_rates["default_rate"] > overall_rate * 1.5]
part_c_report.update({
    "kmeans_best_k": int(best_k),
    "ch_scores_by_k": {str(k): round(v, 2) for k, v in ch_scores.items()},
    "overall_default_rate": round(float(overall_rate), 4),
    "segment_default_rates": segment_default_rates.reset_index().to_dict(orient="records"),
    "segments_over_indexing_on_default": over_indexing.reset_index().to_dict(orient="records"),
})

df.to_csv("outputs/credit_applicants_with_segments.csv", index=False)


Overall default rate: 0.2025
Per-segment default rate:
         n_applicants  default_rate
segment                            
0                 187        0.3209
1                 213        0.0986


In [19]:
with open("outputs/part_c_report.json", "w") as f:
    json.dump(part_c_report, f, indent=2)